 Calidad, Limpieza y Preparación de Datos
En este notebook aplicamos las transformaciones necesarias para corregir las anomalías detectadas en la inspección inicial. Cada decisión está respaldada por evidencia y queda registrada de forma trazable en el log del proyecto.

In [1]:
import pandas as pd
import numpy as np

# Cargar dataset original
df = pd.read_json("../data/raw/streaming_users_dirty.json")

# Guardar estado inicial para el cálculo de retención
filas_iniciales = len(df)
nulos_iniciales = df.isnull().sum().sum()

print(f"Estado inicial: {filas_iniciales} filas y {nulos_iniciales} valores nulos.")

Estado inicial: 8160 filas y 753 valores nulos.


 Eliminación de Registros Duplicados
**Evidencia observada:** Se detectaron 126 registros completamente idénticos en la inspección inicial.
**Acción:** Aplicamos la eliminación de duplicados manteniendo la primera ocurrencia.
**Impacto esperado:** Reducir el sesgo y asegurar que cada fila represente a un usuario único.

In [2]:
# Eliminar duplicados
df = df.drop_duplicates()

# Verificar impacto
filas_despues_dup = len(df)
nulos_despues_dup = df.isnull().sum().sum()
retencion_dup = (filas_despues_dup / filas_iniciales) * 100

print(f"Filas actuales: {filas_despues_dup} (Se eliminaron {filas_iniciales - filas_despues_dup} filas).")
print(f"Porcentaje de retención de datos: {retencion_dup:.2f}%")

Filas actuales: 8034 (Se eliminaron 126 filas).
Porcentaje de retención de datos: 98.46%


 Registro de la primera transformación en el Log (ETL)
Procedemos a inicializar y guardar este primer paso en nuestro archivo `pipeline_log.csv` respetando la estructura obligatoria de la cátedra.

In [7]:
import os

# Crear el DataFrame del log con la estructura obligatoria
log_data = {
    "Paso": [1],
    "Descripción": ["Eliminación de 126 registros duplicados identificados en la inspección inicial"],
    "Filas": [filas_despues_dup],
    "Nulos": [nulos_despues_dup],
    "Retención (%)": [round(retencion_dup, 2)]
}

df_log = pd.DataFrame(log_data)

# Asegurar que la carpeta logs exista
os.makedirs("../logs", exist_ok=True)

# Guardar en la carpeta logs de forma estándar
df_log.to_csv("../logs/pipeline_log.csv", index=False)
print("¡Primer paso registrado con éxito en logs/pipeline_log.csv!")

¡Primer paso registrado con éxito en logs/pipeline_log.csv!


 Imputación y Tratamiento de Valores Faltantes
**Evidencia observada:** Tres variables presentan nulos en proporciones inferiores al 4% (`monthly_watch_time_mins` con 2.40%, `favorite_genre` con 2.99%, y `last_login_date` con 3.98%).
**Acción aplicada:**
1. Imputamos la variable numérica de tiempo con su mediana.
2. Reemplazamos los nulos de la variable categórica de género por la etiqueta explicativa "Unknown".
3. Eliminamos las filas con nulos en la fecha de login por tratarse de un porcentaje marginal.

In [8]:
# 1. Imputar tiempo de visualización con la mediana
mediana_tiempo = df['monthly_watch_time_mins'].median()
df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].fillna(mediana_tiempo)

# 2. Imputar género favorito con "Unknown"
df['favorite_genre'] = df['favorite_genre'].fillna('Unknown')

# 3. Eliminar filas con nulos en la fecha de login
df = df.dropna(subset=['last_login_date'])

# Verificar impacto final
filas_finales = len(df)
nulos_finales = df.isnull().sum().sum()
retencion_final = (filas_finales / filas_iniciales) * 100

print(f"Tratamiento completado con éxito.")
print(f"Estado final: {filas_finales} filas restantes y {nulos_finales} nulos totales.")
print(f"Porcentaje de retención total del dataset: {retencion_final:.2f}%")

Tratamiento completado con éxito.
Estado final: 7714 filas restantes y 0 nulos totales.
Porcentaje de retención total del dataset: 94.53%


 Exportación del Dataset Procesado y Registro en Log ETL Final
Procedemos a almacenar el conjunto de datos limpio en la ubicación obligatoria `data/processed/` y registramos el segundo hito actualizando el archivo de log.

In [9]:
# Asegurar que la carpeta processed exista (fuera de raw)
os.makedirs("../data/processed", exist_ok=True)

# Guardar el dataset limpio en formato JSON
df.to_json("../data/processed/streaming_users_clean.json", orient="records", indent=4)

# Crear la nueva fila de datos para el log
log_paso_2 = pd.DataFrame({
    "Paso": [2],
    "Descripción": ["Tratamiento de nulos: imputación de mediana en tiempo, 'Unknown' en género y eliminación en fechas"],
    "Filas": [filas_finales],
    "Nulos": [nulos_finales],
    "Retención (%)": [round(retencion_final, 2)]
})

# Cargar el log existente, concatenar el nuevo paso y sobreescribir
df_log_actual = pd.read_csv("../logs/pipeline_log.csv")
df_log_completo = pd.concat([df_log_actual, log_paso_2], ignore_index=True)
df_log_completo.to_csv("../logs/pipeline_log.csv", index=False)

print("¡Dataset guardado en data/processed/ y log actualizado correctamente!")

¡Dataset guardado en data/processed/ y log actualizado correctamente!
